In [77]:
import os


In [78]:
os.getcwd()


'C:\\Users\\haari\\OneDrive\\Desktop\\aiml 2026\\Kidney-Disease-Classification-Deep-Learning-Project'

In [79]:
os.chdir(r"C:\Users\haari\OneDrive\Desktop\aiml 2026\Kidney-Disease-Classification-Deep-Learning-Project")


In [80]:
os.getcwd()

'C:\\Users\\haari\\OneDrive\\Desktop\\aiml 2026\\Kidney-Disease-Classification-Deep-Learning-Project'

In [81]:
import dagshub
dagshub.init(repo_owner='haarini-31', repo_name='Kidney-Disease-Classification-Deep-Learning-Project', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

2026-02-21 13:33:38,977 : INFO : httpx : HTTP Request: GET https://dagshub.com/api/v1/repos/haarini-31/Kidney-Disease-Classification-Deep-Learning-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "haarini-31/Kidney-Disease-Classification-Deep-Learning-Project"

2026-02-21 13:33:38,983 : INFO : dagshub : Initialized MLflow to track repo "haarini-31/Kidney-Disease-Classification-Deep-Learning-Project"


Repository haarini-31/Kidney-Disease-Classification-Deep-Learning-Project initialized!

2026-02-21 13:33:38,984 : INFO : dagshub : Repository haarini-31/Kidney-Disease-Classification-Deep-Learning-Project initialized!


2026/02/21 13:33:40 INFO mlflow.tracking._tracking_service.client: 🏃 View run agreeable-lamb-550 at: https://dagshub.com/haarini-31/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/801223c120df4869bd8a700aeec427b8.
2026/02/21 13:33:40 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/haarini-31/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0.


In [82]:
import tensorflow as tf

In [83]:
model=tf.keras.models.load_model( r"C:\Users\haari\OneDrive\Desktop\aiml 2026\Kidney-Disease-Classification-Deep-Learning-Project\artifacts\training\model.h5")

In [ ]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path 
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [85]:
import sys
from pathlib import Path

# get project root (folder that contains 'src')
PROJECT_ROOT = Path(r"C:\Users\haari\OneDrive\Desktop\aiml 2026\Kidney-Disease-Classification-Deep-Learning-Project")

# add src to python path
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PYTHONPATH set correctly ✅")
print(sys.path[0])


PYTHONPATH set correctly ✅
C:\Users\haari\OneDrive\Desktop\aiml 2026\Kidney-Disease-Classification-Deep-Learning-Project\src


In [86]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [87]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.config_dir = Path(self.config.artifacts_root)
        create_directories([self.config_dir])

    def get_evaluation_config(self) -> EvaluationConfig:
        evaluation_config = EvaluationConfig(
            path_of_model=Path(self.config.training.trained_model_path),
            training_data=Path(self.config.training.training_data),
            all_params=self.params,
            mlflow_uri="https://dagshub.com/haarini-31/Kidney-Disease-Classification-Deep-Learning-Project.mlflow",
            params_image_size=self.params.image_size,
            params_batch_size=self.params.batch_size
        )
        return evaluation_config

In [88]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [89]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import mlflow
from urllib.parse import urlparse
from pathlib import Path

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(rescale=1./255, validation_split=0.30)

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    def evaluate(self):
        # Load trained model correctly
        self.model = load_model(self.config.path_of_model)

        # Prepare validation data
        self._valid_generator()

        # Evaluate model
        self.score = self.model.evaluate(self.valid_generator)

    def save_score(self):
        score = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }
        save_json(path=Path("scores.json"), data=score)

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(self.config.mlflow_uri).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("loss", self.score[0])
            mlflow.log_metric("accuracy", self.score[1])

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [90]:
config = ConfigurationManager()
evaluation_config = config.get_evaluation_config()
evaluation = Evaluation(config=evaluation_config)
evaluation.evaluate()
evaluation.save_score()
evaluation.log_into_mlflow()

2026-02-21 13:33:41,712 : INFO : cnnClassifier : yaml file: configs\config.yaml loaded successfully
2026-02-21 13:33:41,715 : INFO : cnnClassifier : yaml file: params.yaml loaded successfully
2026-02-21 13:33:41,716 : INFO : cnnClassifier : created directory at: artifacts
Found 3733 images belonging to 1 classes.
  1/234 [..............................] - ETA: 5:40 - loss: 92.1501 - accuracy: 0.3750

KeyboardInterrupt: 

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="VGG16Model",
    alias="production",
    version="1"
)

print("Alias set successfully!")

Alias set successfully!
